In [1]:
'''
task: FOL classification based on NL + CLINGO text
model: FOLIO NL + CLINGO t5-large
dataset: SemEval-2026
evaluation: inference on practice test dataset
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# load test set
test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/subtask1_test.csv")

# transform data to acceptable format for model training
test_df = pd.DataFrame(test)

test_df = test_df.applymap(str)

/tmp/ipython-input-981353079.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  test_df = test_df.applymap(str)


In [3]:
# start preparing for QA pipeline
!pip install datasets
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 50.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00


In [4]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import T5Tokenizer, T5Model, T5ForConditionalGeneration, T5TokenizerFast

import warnings
warnings.filterwarnings("ignore")

In [5]:
MODEL = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/flan-t5-large_folio_nl_clingo_model")
TOKENIZER = T5TokenizerFast.from_pretrained("/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/flan-t5-large_folio_nl_clingo_tokenizer")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [6]:
#torch.cuda.empty_cache()
MODEL.to('cuda')

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

In [7]:
# evaluation metrics

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report


def predict_answer(subject, ref_relation=None):
    inputs = TOKENIZER(subject, return_tensors="pt").to(MODEL.device)

    outputs = MODEL.generate(input_ids=inputs["input_ids"], max_new_tokens=10)

    predicted_relation = TOKENIZER.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

    print("syllogism: \n", subject)
    print("true label: \n", ref_relation)
    print("predicted label: \n", predicted_relation)

    return predicted_relation

In [8]:
# test predictions
predictions = []
for index, row in test_df.iterrows():
  prediction_dict = {}
  syl_id = row["id"]
  premises = row["syllogism"] + " " + row["clingo"]

  predicted_label = predict_answer(premises)
  # generate and format prediction json
  prediction_dict['id'] = syl_id
  prediction_dict['validity'] = predicted_label.lower()
  predictions.append(prediction_dict)


syllogism: 
 ∀x (Bikes(x) → ¬Calledcars(x))
∀x (Bike(x) → Vehicle(x))
∃x (Vehicles(x) ∧ Bikes(x))
 forall x (bikes(x) implies not calledcars(x))
forall x (bike(x) implies vehicle(x))
exists x (vehicles(x) and bikes(x))

true label: 
 None
predicted label: 
 False
syllogism: 
 ∃x (Objects(x) ∧ Booksnotdigitalfiles(x))
∀x (Book(x) → Itempages(x))
∃x (Itemspages(x) ∧ Digitalfiles(x))
 exists x (objects(x) and booksnotdigitalfiles(x))
forall x (book(x) implies itempages(x))
exists x (itemspages(x) and digitalfiles(x))

true label: 
 None
predicted label: 
 False
syllogism: 
 ∀x (Object(x) → CanFly(x))
∃x (Boats(x) ∧ ¬CanFly(x))
∃x (Boats(x) ∧ Notobjects(x))
 forall x (object(x) implies canfly(x))
exists x (boats(x) and not canfly(x))
exists x (boats(x) and notobjects(x))

true label: 
 None
predicted label: 
 False
syllogism: 
 ∀x (Fruit(x) → Food(x))
∀x (Nothing(x) → Vegetablefood(x))
∀x (Vegetable(x) → Fruit(x))
 forall x (fruit(x) implies food(x))
forall x (nothing(x) implies vegetablef

In [9]:
# export predictions
import json

with open('predictions.json', 'w') as f:
    json.dump(predictions, f)